## 04. scratchpad

### 1. Data Upload

In [567]:
import pandas as pd
import os
from collections import defaultdict, Counter
import re
import time
from tqdm.notebook import tqdm
import copy

In [2]:
def upload_to_dataframe(root, files, num_lines):
    path_eng, path_pol = [root+f for f in files]
    data = defaultdict(list)
    
    with open (path_eng, encoding="utf_8") as f_eng, open (path_pol, encoding="utf_8") as f_pol:
        for _ in range(num_lines):
            data['eng_text'].append(f_eng.readline().strip())
            data['pol_text'].append(f_pol.readline().strip())
    return pd.DataFrame(data)

In [3]:
root = "../data/opus_opensub/en-pl.txt/"
files = ('OpenSubtitles.en-pl.en', '/OpenSubtitles.en-pl.pl')
df = upload_to_dataframe(root, files, 600000)

### 2. EDA & Sanitazation

#### 2.1 English Non-Ascii Sentences

In [4]:
def nonascii_list(df_series, is_pol):
    if is_pol:
        pl_chars = set("ąęćłńóśźżĄĘĆŁŃÓŚŹŻ")
        requir = lambda x: x.isascii() or x in pl_chars
    else:
        requir = lambda x: x.isascii()
        
    text_all = ''.join(df_series)
    unique_dct = {x for x in text_all if not requir(x)}
    return sorted(list(unique_dct))

In [5]:
print(nonascii_list(df['eng_text'], False))

['\x80', '\x81', '\x82', '\x83', '\x84', '\x85', '\x88', '\x8b', '\x8c', '\x8e', '\x8f', '\x91', '\x94', '\x96', '\x98', '\x99', '\x9c', '\x9d', '\xa0', '¡', '¢', '£', '¤', '¥', '¦', '§', '¨', '©', 'ª', '¬', '\xad', '°', '±', '²', '³', '´', 'µ', '¶', '·', '¸', '¹', 'º', '¼', '½', '¾', '¿', 'Á', 'Â', 'Ã', 'Ä', 'Å', 'Ç', 'É', 'Ê', 'Ì', 'Ï', 'Ð', 'Ñ', 'Ô', 'Ö', 'Ø', 'Ü', 'Ý', 'Þ', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'ç', 'è', 'é', 'ê', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ó', 'ô', 'ö', 'ù', 'ú', 'û', 'ü', 'ý', 'þ', 'Ă', 'ď', 'ę', 'Ł', 'ń', 'ő', 'œ', 'ś', 'Š', 'š', 'Ј', 'С', 'й', '،', 'أ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ح', 'ر', 'س', 'ش', 'ع', 'غ', 'ف', 'ك', 'ل', 'م', 'ن', 'و', 'ي', 'َ', 'ُ', 'ِ', 'ّ', 'ْ', '\u200b', '‒', '–', '—', '‚', '€', '™', '─', '☻', '♥', '♪', '慹', '拢', '檛', '鈥', 'ﬁ', 'ﬂ']


In [6]:
is_ascii = lambda text: text.isascii()
maska = df['eng_text'].apply(is_ascii)
df_non_ascii = df[~maska].copy().reset_index(drop=True)
print(df_non_ascii.shape)
df_non_ascii.head()

(4595, 2)


,eng_text,pol_text
0,¿You had never seen A crab suedes? Them there ...,- Nigdy wcześniej nie widziałaś raków?
1,"He hears ¿, where you live?",Nigdy wcześniej nie byłam w lesie.
2,"Boys ¿, they are there?",Jesteście tam chłopaki?
3,- Te atrapé. - No es gracioso. Your he catches...,- Ale twoje mina była.
4,"Your also ¿, it is not like that?","- Tak, niezła jest"


In [7]:
df_2 = df[maska].reset_index(drop=True)
df_2.shape

(595405, 2)

#### 2.2 Polish Non-Ascii Sentences

In [8]:
pl_chars = set("ąęćłńóśźżĄĘĆŁŃÓŚŹŻ")
requir = lambda let: not let.isascii() and let not in pl_chars
maska = df_2['pol_text'].apply(lambda snt: any([requir(let) for let in snt]))
df_non_ascii = df_2[maska].copy().reset_index(drop=True)
print(df_non_ascii.shape)
df_non_ascii.sample(10)

(1227, 2)


,eng_text,pol_text
170,"We're going back by water, Kali, using Mr. Tho...","Wracamy drogą wodną, Kalí, czółnem pana Thornt..."
1162,"Yes, a walk up the avenue's What we'll take",¶ Więc idziemy sobie ulicą ¶ Aż dotrzemy kiedy...
651,"If you like business, you can pick any job and...","Ješli tak, može pan sobie zajac dowolnie wybra..."
423,He and his little four page paper against that...,Czterostronnicowa broszura przeciwko syndykato...
734,I did not say that Senator Paine was one of th...,"Nie powiedzialem, že jednym z nich byl senator..."
130,""""" Tis a pity that some of our compatriots are...","""Szkoda, że ​​niektórzy z naszych rodaków... s..."
916,She sure made this disappearance act look good.,Z pewnością ​​ten akt zniknięcia wyglądał dobrze.
703,You can't quit now.,Nie možesz teraz odjesc.
1103,The fella with an umbrella,¶ Facetowi z parasolem
1206,And you'll find that you're In the rotogravure,¶ I zobaczysz siebie w gazetach


In [9]:
df_3 = df_2[~maska].reset_index(drop=True)
print(df_3.shape)
df_3.sample(5)

(594178, 2)


,eng_text,pol_text
406669,It was her laugh in the gallery that woke me.,To jej śmiech w galerii mnie obudził.
392337,But you better keep an eye out for rattlers an...,"Ale uważaj na grzechotniki , kojoty i skunksy."
456792,See you tomorrow.,Do jutra.
210340,"Henry Higgins, do you see this?","Henry Higgins, Widzisz?"
214672,"Here, here.","Hej, hej."


In [10]:
print(nonascii_list(df_3['eng_text'], False))
print(nonascii_list(df_3['pol_text'], True))

[]
[]


In [11]:
df_3.iloc[74182]['eng_text']

'The next time I...'

In [12]:
df_3.iloc[74182]['pol_text']

'Następnym razem...'

#### 2.3 Short Or Empty Examples

In [13]:
mask1 = df_3['eng_text'].str.len() >= 3
mask2 = df_3['pol_text'].str.len() >= 3

In [14]:
df_4 = df_3[mask1 & mask2].reset_index(drop=True)
df_4.head()

,eng_text,pol_text
0,"Previously on ""The Blacklist""...",/W poprzednich odcinkach: /
1,- You want to call your daddy?,- Chcesz zadzwonić do taty?
2,"- Yeah, I want to tell him I'm okay.","- Tak, powiem, że wszystko w porządku."
3,Okay.,Dobrze.
4,Lizzy... Be careful of your husband.,"Lizzy, uważaj na męża."


In [15]:
df_4.shape

(593894, 2)

#### 2.4 Dialog like sentences [starts with - ]

In [16]:
is_dialog = lambda snt: bool(re.match(r"^ *- *", snt))
mask = df_4['eng_text'].apply(is_dialog) | df_4['pol_text'].apply(is_dialog)
df_dialogs = df_4[mask].copy().reset_index(drop=True)
df_dialogs.head()

,eng_text,pol_text
0,- You want to call your daddy?,- Chcesz zadzwonić do taty?
1,"- Yeah, I want to tell him I'm okay.","- Tak, powiem, że wszystko w porządku."
2,- 45 seconds.,Szybciej! 45 sekund.
3,It's okay. It's okay. It's okay.,- Już dobrze.
4,It's okay. You're okay. You're okay.,- Odpoczywaj.


In [17]:
def fix_dialog(snt):
    return re.sub(r"^ *- *", "", snt)

df_4['eng_text'] = df_4['eng_text'].apply(fix_dialog).reset_index(drop=True)
df_4['pol_text'] = df_4['pol_text'].apply(fix_dialog).reset_index(drop=True)
df_4.sample(5)

,eng_text,pol_text
377519,"It's that rubber band. Pulaski's, I'd know it ...",Wszędzie poznałbym tę gumkę od Pułaskiej.
274008,What? - You`d better get on the roof.,Wchodźcie obydwaj na dach!
147926,That's awful pretty.,To jest piękne.
345672,"Oh, a little of this, a little of that. Not mu...","Trochę tego, trochę tamtego, ale niewiele czeg..."
420191,Didn't he give you his name?,Nie podał imienia?


#### 2.5 Broken Dialog like sentences [somehwere has -]

# Do naprawy (lepszy re przed tokenziacja)

In [18]:
broken_dialog = lambda snt: bool(re.match(r".* +- +.*", snt))
mask1 = df_4['eng_text'].apply(broken_dialog)
mask2 = df_4['pol_text'].apply(broken_dialog)

In [19]:
df_4[mask1 | mask2].sample(5)

,eng_text,pol_text
508203,Not bad. - With bath?,"Ani tu dobrze, ani źle."
242568,Silly of me - I adore youth!,Głupio z mojej strony - uwielbiam młodość.
532936,"Not a trace of them, sir. - We're searching-","Nie ma po nich śladu, sir."
508437,All he wanted was for me to get his girl out o...,"Chciał, żebym wydostał jego dziewczynę z więzi..."
157689,I've got everything fixed up now for an attack...,Mam wszystko przygotowane do ataku na Floss Va...


In [20]:
df_5 = df_4[~mask1 & ~mask2].reset_index(drop=True)

In [21]:
broken_dialog = lambda snt: bool(re.match(r".*-.*", snt))

In [22]:
df_5.sample(5)

,eng_text,pol_text
61228,You know what men are like on a Saturday.,"Wiesz, jak to jest w soboty z mężczyznami."
432733,Right.,Zgoda.
55580,"Now, let's talk about you.",A teraz pomówmy trochę o tobie.
43646,WE'LL HAVE TO STOP AND GET MY THINGS.,Musimy się zatrzymać i zabrać moje rzeczy.
260663,"Oh, this is something different.","O, to co innego."


#### 2.6 Different Last Case

In [23]:
mask = df_5['eng_text'].str[-1] != df_5['pol_text'].str[-1]
mask.sum()

57558

In [24]:
df_6 = df_5[~mask].reset_index(drop=True)

#### 2.7 Text length difference

In [25]:
mask = df_6['eng_text'].str.split
df_6['len_ratio'] = df_6['eng_text'].str.split().str.len() / df_6['pol_text'].str.split().str.len()
df_6.head()

,eng_text,pol_text,len_ratio
0,You want to call your daddy?,Chcesz zadzwonić do taty?,1.500000
1,"Yeah, I want to tell him I'm okay.","Tak, powiem, że wszystko w porządku.",1.333333
2,Okay.,Dobrze.,1.000000
3,Lizzy... Be careful of your husband.,"Lizzy, uważaj na męża.",1.500000
4,I can only lead you to the truth.,/Mogę naprowadzić cię na prawdę.,1.600000


In [26]:
thres_left = df_6["len_ratio"].quantile(0.05)
thres_right = df_6["len_ratio"].quantile(0.95)
thres_left, thres_right

(0.75, 2.5)

In [27]:
mask = (df_6["len_ratio"] >= thres_left) & (df_6["len_ratio"] <= thres_right)
mask.sum()

472209

In [28]:
df_7 = df_6[mask].reset_index(drop=True)
df_7.sample(5)

,eng_text,pol_text,len_ratio
164175,"In Heaven's name, why?","Na niebiosa, dlaczego?",1.333333
359981,"Sloan, keep out of this.",Nie mieszaj się do tego.,1.000000
197603,You try to ruin my business?,Chcesz zrujnować mój interes?,1.500000
62736,I was going to say it a little further along.,Chciałem to powiedzieć nieco później.,2.000000
215229,"Would you say a few words, Casy?","Może powiesz parę słów, Casy?",1.400000


In [29]:
df_7.sample(5)

,eng_text,pol_text,len_ratio
375375,I have seen it in the newspaper. Who beat Chue...,To pan pokonał pana Chuen w zawodach policji.,1.75
244631,Which way has the slave taken?,Jaką drogę obrał niewolnik?,1.50
311414,"I've heard of you, Monsieur Inspector.","Słyszałem o panu, inspektorze.",1.50
313327,A photographic ship accompanies missions to Ki...,W lotach nad Kiską bombowcom towarzyszy samolo...,0.88
446360,"Well, let's go.",No to jedźmy.,1.00


#### 2.8 Sentences starting with * fix

In [30]:
fix_star = lambda snt: re.sub(r"^ *[*].*", "", snt)
df_7['eng_text'] = df_7['eng_text'].apply(fix_star)
df_7['pol_text'] = df_7['pol_text'].apply(fix_star)

#### 2.9 Action Comments

In [31]:
def mask_distinct(df, col1, col2, chars):
    chars_escaped = "".join(re.escape(c) for c in chars)
    templ = f"[{chars_escaped}]"

    m1 = df[col1].str.contains(templ, regex=True, na=False)
    m2 = df[col2].str.contains(templ, regex=True, na=False)

    return m1 ^ m2

chars = '[]()/\*+#$"️'
mask = mask_distinct(df_7, "eng_text", "pol_text", chars)

df_8 = df_7[~mask].reset_index(drop=True)
df_8.shape

(467550, 3)

In [32]:
df_8.sample(5)

,eng_text,pol_text,len_ratio
324578,"With love, Gloria.","Pozdrowienia, Gloria.",1.500000
318905,It is a school for unfortunate orphans.,To jest szkoła dla nieszczęśliwych sierot.,1.166667
122691,"Van, come home.","Van, wróć do domu.",0.750000
365856,No.,Nie.,1.000000
277049,Go back and wait.,Poczekaj tam na mnie.,1.000000


#### 2.10  " ` " ---> " ' " & del " ~ "

In [33]:
wanted_case = '`'
mask = df_8["eng_text"].apply(lambda x: wanted_case in x)
df_8[mask]

,eng_text,pol_text,len_ratio
214145,"Revolution had broken out, her diplomats sued ...","Rewolucja na tyłach frontu, dąży do zawarcia p...",1.277778
214152,We`ll examine it.,Zbadajmy go.,1.500000
214157,What do you think you`re doing?,Wstawaj! Co ty robisz?,1.500000
214161,Where`s your hand grenade?,Gdzie masz swój granat?,1.000000
214163,Let `em have it!,Niech je poczują!,1.333333
...,...,...,...
214943,We don`t want to hate one another.,Bez nienawiści ani pogardy.,1.750000
214944,There`s room for everyone.,Każdy ma swe miejsce.,1.000000
214947,"Greed has poisoned men`s souls, has barricaded...","Chciwość zatruła nasze dusze, wzniosła mury ni...",1.333333
214963,You don`t hate.,Przestańcie nienawidzieć.,1.500000


In [34]:
df_8["eng_text"] = df_8["eng_text"].str.replace("`", "'", regex=False)
df_8["pol_text"] = df_8["pol_text"].str.replace("`", "'", regex=False)

In [35]:
df_8["eng_text"] = df_8["eng_text"].str.replace("~", "", regex=False)
df_8["pol_text"] = df_8["pol_text"].str.replace("~", "", regex=False)

#### 2.11 Del rows with specific case

In [36]:
def del_row_with_char(df, eng_col, pol_col, chars):
    df_proc = df.copy()
    for char in chars:
        mask1 = df_proc[eng_col].apply(lambda x: char in x)
        mask2 = df_proc[pol_col].apply(lambda x: char in x)
        df_proc = df_proc[~(mask1 | mask2)].reset_index(drop=True)
    return df_proc
    

chars = "=+*@#;|_}{"
df_9 = del_row_with_char(df_8, "eng_text", "pol_text", chars)

#### 2.12 Del rows with dialog -

In [37]:
mask1 = df_9["eng_text"].apply(lambda x: bool(re.search(r' +-[^ ]+', x)))
mask2 = df_9["pol_text"].apply(lambda x: bool(re.search(r' +-[^ ]+', x)))
df_data = df_9[~(mask1 | mask2)].reset_index(drop=True)

#### 3. Data Tokenization

In [38]:
uniq_cases = set("".join(df_data["eng_text"].astype(str)))
tab1 = sorted(list(uniq_cases), reverse=True)
print(tab1)

['z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '$', '"', '!', ' ']


In [39]:
uniq_cases = set("".join(df_data["pol_text"].astype(str)))
tab2 = sorted(list(uniq_cases), reverse=True)
print(tab2)

['ż', 'Ż', 'ź', 'Ź', 'ś', 'Ś', 'ń', 'Ń', 'ł', 'Ł', 'ę', 'Ę', 'ć', 'Ć', 'ą', 'Ą', 'ó', 'Ó', 'z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '$', '"', '!', ' ']


In [40]:
def tokenize_eng(snt):
    snt = snt.lower()
    snt = re.sub("(?<! )'(?! )", r" '", snt)
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

In [41]:
def tokenize_pol(snt):
    snt = snt.lower()
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

In [42]:
df_data = df_data.drop(['len_ratio'], axis=1)

#### 3.1 Splitting and trimming on quantile

In [43]:
df_data['eng_split'] = df_data['eng_text'].apply(tokenize_eng)
df_data['pol_split'] = df_data['pol_text'].apply(tokenize_pol)
df_data.sample(10)

,eng_text,pol_text,eng_split,pol_split
322714,I wanted my movements accounted for up to the ...,"Chciałem, aby wszystkie moje kroki były widocz...","[i, wanted, my, movements, accounted, for, up,...","[chciałem, ,, aby, wszystkie, moje, kroki, był..."
301363,One of the things you do not know is that Colo...,"Jedną z rzeczy, których nie wiesz, jest to że ...","[one, of, the, things, you, do, not, know, is,...","[jedną, z, rzeczy, ,, których, nie, wiesz, ,, ..."
402335,You should know better than to give in to your...,Nie powinnaś ulegać swoim impulsom.,"[you, should, know, better, than, to, give, in...","[nie, powinnaś, ulegać, swoim, impulsom, .]"
209673,Thank you.,Dziękuję.,"[thank, you, .]","[dziękuję, .]"
337890,It's not me this time...,Tym razem to nie ja...,"[it, 's, not, me, this, time, ., ., .]","[tym, razem, to, nie, ja, ., ., .]"
151666,"Here you are, Mrs Gilbert.","Tutaj, pani Gilbert.","[here, you, are, ,, mrs, gilbert, .]","[tutaj, ,, pani, gilbert, .]"
216680,Get out of the way.,Zejdźcie z drogi.,"[get, out, of, the, way, .]","[zejdźcie, z, drogi, .]"
2865,"And now, my son, let us assist in carrying you...","A teraz, mój synu, pozwól mi zdjąć z ciebie to...","[and, now, ,, my, son, ,, let, us, assist, in,...","[a, teraz, ,, mój, synu, ,, pozwól, mi, zdjąć,..."
120090,I've been running across that log ever since I...,Biegałem przez ten dziennik odkąd byłem po kol...,"[i, 've, been, running, across, that, log, eve...","[biegałem, przez, ten, dziennik, odkąd, byłem,..."
112881,How do you do? Excuse me.,Generał Pompil o Montezuma .,"[how, do, you, do, ?, excuse, me, .]","[generał, pompil, o, montezuma, .]"


# TODO - zamien lt na It

In [388]:
df_data[df_data['eng_split'].apply(lambda x: 'lt' in x)]

,eng_text,pol_text,eng_split,pol_split
9576,lt was the only thing they had.,"To było jedyne, jakie mieli wolne.","[lt, was, the, only, thing, they, had, .]","[to, było, jedyne, ,, jakie, mieli, wolne, .]"
9606,lt was your fault you put the thing under the ...,"To twoja wina, bo położyłeś go pod łóżko.","[lt, was, your, fault, you, put, the, thing, u...","[to, twoja, wina, ,, bo, położyłeś, go, pod, ł..."
10725,Beat it. lt's a bull!,"Uciekajmy, to gliniarz!","[beat, it, ., lt, 's, a, bull, !]","[uciekajmy, ,, to, gliniarz, !]"
10729,lt went in your pocket.,Wpadł ci do kieszeni.,"[lt, went, in, your, pocket, .]","[wpadł, ci, do, kieszeni, .]"
10753,lt caught in your pocket.,On utknął w kieszeni...,"[lt, caught, in, your, pocket, .]","[on, utknął, w, kieszeni, ., ., .]"
...,...,...,...,...
407392,Okay. lt's your f uneral.,Dobrze. To twój cyrk.,"[okay, ., lt, 's, your, f, uneral, .]","[dobrze, ., to, twój, cyrk, .]"
407396,lt's important.,To ważne.,"[lt, 's, important, .]","[to, ważne, .]"
407469,"Maybe, maybe not. lt's a pretty story.","Może tak, może nie. Niezła historia.","[maybe, ,, maybe, not, ., lt, 's, a, pretty, s...","[może, tak, ,, może, nie, ., niezła, historia, .]"
407484,lt's unlocked.,Jest otwarta.,"[lt, 's, unlocked, .]","[jest, otwarta, .]"


In [44]:
thres1 = df_data['eng_split'].str.len().quantile(0.99)
mask1 = df_data['eng_split'].str.len() > thres1

thres2 = df_data['pol_split'].str.len().quantile(0.99)
mask2 = df_data['pol_split'].str.len() > thres2
print(mask1.sum(), mask2.sum(), (mask1 | mask2).sum())

df_data = df_data[~(mask1 | mask2)].reset_index(drop=True)

4113 3960 5510


#### 3.2 BPE Init. Uniq_freq & Rozklad chars

In [45]:
uniq_freq_eng = Counter(df_data['eng_split'].explode())
uniq_freq_pol = Counter(df_data['pol_split'].explode())

In [46]:
len(uniq_freq_eng), len(uniq_freq_pol)

(40087, 113603)

In [377]:
char_factor = lambda word: [x for x in word] + ['_']
vocab_chars_eng = [[char_factor(k), v] for k, v in uniq_freq_eng.most_common()]
vocab_chars_pol = [[char_factor(k), v] for k, v in uniq_freq_pol.most_common()]
print(len(vocab_chars_eng), vocab_chars_eng[:5], '\n')
print(len(vocab_chars_pol), vocab_chars_pol[:5]) 

40087 [[['.', '_'], 435802], [[',', '_'], 177291], [['y', 'o', 'u', '_'], 134548], [['i', '_'], 122767], [['t', 'h', 'e', '_'], 89859]] 

113603 [[['.', '_'], 413780], [[',', '_'], 183948], [['?', '_'], 83444], [['n', 'i', 'e', '_'], 82694], [['t', 'o', '_'], 53190]]


#### 3.3 BPE Tokenizer most common pairs

In [527]:
class BPETokenizer():
    def __init__(self, vocab_chrs, vocab_size, is_tgt):
        self.vocab_chrs = vocab_chrs
        self.vocab_size = vocab_size
        self.vocab = self.vocab_init(vocab_chrs, is_tgt)
        self.base_size = len(self.vocab)

    def vocab_init(self, vocab_chrs, is_tgt):
        vocab = {'<pad>': 0, '<unk>': 1, '<eos>': 2, '_': 3}
        if is_tgt:
            vocab['<bos>'] = 4
            
        uniq_toks = {tok for toks, _ in vocab_chrs for tok in toks}
        uniq_toks.remove('_')
        for i, tok in enumerate(uniq_toks, len(vocab)):
            vocab[tok] = i
        return vocab                

    def train_bpe(self):
        for i in tqdm(range(self.base_size, self.vocab_size)):
            pair = self.most_common_pair()
            self.vocab[' '.join(pair)] = i
            self.replace_pairs(pair)

    def replace_pairs(self, pair):
        pair_con = f" {pair[0]}{pair[1]} "
        p_1, p_2 = map(re.escape, pair)
        
        for i, (toks, _) in enumerate(self.vocab_chrs):
            if pair in zip(toks, toks[1:]):
                self.vocab_chrs[i][0] = re.sub(rf"\s{p_1}\s{p_2}\s", pair_con, f" {' '.join(toks)} ").split()

    def most_common_pair(self):
        pair_counts = defaultdict(int)
        for toks, freq in self.vocab_chrs:
            for pair in zip(toks, toks[1:]):
                pair_counts[pair] += freq
        return max(pair_counts, key=pair_counts.get)


### 1. tokenizer

In [381]:
tokenizer_eng = BPETokenizer(vocab_chars_eng, 8000, False)
t = time.time()
tokenizer_eng.train_bpe()
print(f"{time.time()-t:.2f} seconds")

  0%|          | 0/7945 [00:00<?, ?it/s]

500.43 seconds


### 4. Testing Pair Count Func.

In [568]:
def replace_test(pair, vocab):
    indices = []
    vocab_chrs = copy.deepcopy(vocab)
    pair_con = f" {pair[0]}{pair[1]} "
    p_1, p_2 = map(re.escape, pair)
        
    for i, (toks, _) in enumerate(vocab_chrs):
        if pair in zip(toks, toks[1:]):
            vocab_chrs[i][0] = re.sub(rf"\s{p_1}\s{p_2}\s", pair_con, f" {' '.join(toks)} ").split()
            indices.append(i)
    return vocab_chrs, indices

#### 4.1 Initial Version

In [437]:
def func_1(vocab):
    pair_counts = defaultdict(int)
    for toks, freq in vocab:
        for pair in zip(toks, toks[1:]):
            pair_counts[pair] += freq
    return max(pair_counts, key=pair_counts.get), pair_counts

In [434]:
test_vocab = [[char_factor(k), v] for k, v in uniq_freq_eng.most_common()]
t = time.time()
for _ in range(100):
    func_1(test_vocab)
    pass
print(f"{time.time()-t:.2f} seconds")

4.82 seconds


In [569]:
test_vocab1 = [[char_factor(k), v] for k, v in uniq_freq_eng.most_common()]
pair1, dct_c1 = func_1(test_vocab1)
dct_c1 = dict(sorted(dct_c1.items(), key=lambda x: x[1], reverse=True))
print(f"Pair 1: {pair1}")

test_vocab2, ids = replace_test(pair1, test_vocab1)
pair2, dct_c2 = func_1(test_vocab2)
dct_c2 = dict(sorted(dct_c2.items(), key=lambda x: x[1], reverse=True))
print(f"Pair 2: {pair2}")

Pair 1: ('e', '_')
Pair 2: ('.', '_')


In [573]:
kurwa_1 = [test_vocab1[i] for i in ids]
kurwa_2 = [test_vocab2[i] for i in ids]

In [626]:
kurwa_1[:10]

[[['t', 'h', 'e', '_'], 89859],
 [['m', 'e', '_'], 29977],
 [['h', 'e', '_'], 26149],
 [['w', 'e', '_'], 23370],
 [['b', 'e', '_'], 20137],
 [['h', 'a', 'v', 'e', '_'], 19830],
 [["'", 'r', 'e', '_'], 15483],
 [['h', 'e', 'r', 'e', '_'], 15059],
 [['a', 'r', 'e', '_'], 14431],
 [['t', 'h', 'e', 'r', 'e', '_'], 14310]]

In [627]:
kurwa_2[:10]

[[['t', 'h', 'e_'], 89859],
 [['m', 'e_'], 29977],
 [['h', 'e_'], 26149],
 [['w', 'e_'], 23370],
 [['b', 'e_'], 20137],
 [['h', 'a', 'v', 'e_'], 19830],
 [["'", 'r', 'e_'], 15483],
 [['h', 'e', 'r', 'e_'], 15059],
 [['a', 'r', 'e_'], 14431],
 [['t', 'h', 'e', 'r', 'e_'], 14310]]

In [606]:
vocab_pair_1 = [[list(zip(tab, tab[1:])), c] for tab, c in test_vocab1]
vocab_pair_1 = [vocab_pair_1[i] for i in ids]
vocab_pair_1

[[[('t', 'h'), ('h', 'e'), ('e', '_')], 89859],
 [[('m', 'e'), ('e', '_')], 29977],
 [[('h', 'e'), ('e', '_')], 26149],
 [[('w', 'e'), ('e', '_')], 23370],
 [[('b', 'e'), ('e', '_')], 20137],
 [[('h', 'a'), ('a', 'v'), ('v', 'e'), ('e', '_')], 19830],
 [[("'", 'r'), ('r', 'e'), ('e', '_')], 15483],
 [[('h', 'e'), ('e', 'r'), ('r', 'e'), ('e', '_')], 15059],
 [[('a', 'r'), ('r', 'e'), ('e', '_')], 14431],
 [[('t', 'h'), ('h', 'e'), ('e', 'r'), ('r', 'e'), ('e', '_')], 14310],
 [[("'", 'v'), ('v', 'e'), ('e', '_')], 11579],
 [[('l', 'i'), ('i', 'k'), ('k', 'e'), ('e', '_')], 11372],
 [[('s', 'h'), ('h', 'e'), ('e', '_')], 10741],
 [[('c', 'o'), ('o', 'm'), ('m', 'e'), ('e', '_')], 10653],
 [[('s', 'e'), ('e', 'e'), ('e', '_')], 9619],
 [[('o', 'n'), ('n', 'e'), ('e', '_')], 8831],
 [[('t', 'a'), ('a', 'k'), ('k', 'e'), ('e', '_')], 6657],
 [[('w', 'h'), ('h', 'e'), ('e', 'r'), ('r', 'e'), ('e', '_')], 6175],
 [[('t', 'i'), ('i', 'm'), ('m', 'e'), ('e', '_')], 5538],
 [[('w', 'e'), ('e', 

In [607]:
vocab_pair_2 = [[list(zip(tab, tab[1:])), c] for tab, c in test_vocab2]
vocab_pair_2 = [vocab_pair_2[i] for i in ids]
vocab_pair_2

[[[('t', 'h'), ('h', 'e_')], 89859],
 [[('m', 'e_')], 29977],
 [[('h', 'e_')], 26149],
 [[('w', 'e_')], 23370],
 [[('b', 'e_')], 20137],
 [[('h', 'a'), ('a', 'v'), ('v', 'e_')], 19830],
 [[("'", 'r'), ('r', 'e_')], 15483],
 [[('h', 'e'), ('e', 'r'), ('r', 'e_')], 15059],
 [[('a', 'r'), ('r', 'e_')], 14431],
 [[('t', 'h'), ('h', 'e'), ('e', 'r'), ('r', 'e_')], 14310],
 [[("'", 'v'), ('v', 'e_')], 11579],
 [[('l', 'i'), ('i', 'k'), ('k', 'e_')], 11372],
 [[('s', 'h'), ('h', 'e_')], 10741],
 [[('c', 'o'), ('o', 'm'), ('m', 'e_')], 10653],
 [[('s', 'e'), ('e', 'e_')], 9619],
 [[('o', 'n'), ('n', 'e_')], 8831],
 [[('t', 'a'), ('a', 'k'), ('k', 'e_')], 6657],
 [[('w', 'h'), ('h', 'e'), ('e', 'r'), ('r', 'e_')], 6175],
 [[('t', 'i'), ('i', 'm'), ('m', 'e_')], 5538],
 [[('w', 'e'), ('e', 'r'), ('r', 'e_')], 4912],
 [[('l', 'i'), ('i', 't'), ('t', 't'), ('t', 'l'), ('l', 'e_')], 4764],
 [[('s', 'o'), ('o', 'm'), ('m', 'e_')], 4544],
 [[('m', 'a'), ('a', 'k'), ('k', 'e_')], 4312],
 [[('m', 'o'),

In [579]:
kurwa_1[:10]

[[['t', 'h', 'e', '_'], 89859],
 [['m', 'e', '_'], 29977],
 [['h', 'e', '_'], 26149],
 [['w', 'e', '_'], 23370],
 [['b', 'e', '_'], 20137],
 [['h', 'a', 'v', 'e', '_'], 19830],
 [["'", 'r', 'e', '_'], 15483],
 [['h', 'e', 'r', 'e', '_'], 15059],
 [['a', 'r', 'e', '_'], 14431],
 [['t', 'h', 'e', 'r', 'e', '_'], 14310]]

In [580]:
kurwa_2[:10]

[[['t', 'h', 'e_'], 89859],
 [['m', 'e_'], 29977],
 [['h', 'e_'], 26149],
 [['w', 'e_'], 23370],
 [['b', 'e_'], 20137],
 [['h', 'a', 'v', 'e_'], 19830],
 [["'", 'r', 'e_'], 15483],
 [['h', 'e', 'r', 'e_'], 15059],
 [['a', 'r', 'e_'], 14431],
 [['t', 'h', 'e', 'r', 'e_'], 14310]]

In [621]:
dct_c2

{('.', '_'): 435802,
 ('t', '_'): 405188,
 ('t', 'h'): 286912,
 ('s', '_'): 279133,
 ('o', 'u'): 241694,
 ('n', '_'): 224509,
 ('d', '_'): 199163,
 ('i', 'n'): 187051,
 (',', '_'): 177291,
 ('e', 'r'): 171377,
 ('y', '_'): 170823,
 ('r', '_'): 169575,
 ('y', 'o'): 161552,
 ('o', '_'): 158570,
 ('a', 'n'): 156511,
 ('h', 'a'): 141491,
 ('u', '_'): 135623,
 ('h', 'e_'): 127255,
 ('i', '_'): 126232,
 ('h', 'e'): 123261,
 ('o', 'n'): 122342,
 ('a', 't'): 120991,
 ('t', 'o'): 108364,
 ('l', '_'): 107287,
 ('l', 'l'): 104796,
 ('i', 't'): 104719,
 ('n', 'g'): 99478,
 ('r', 'e_'): 92025,
 ('g', '_'): 89713,
 ('o', 'r'): 88425,
 ('h', 'i'): 86574,
 ('n', 'd'): 86449,
 ('i', 's'): 85924,
 ('e', 'n'): 85895,
 ('?', '_'): 84640,
 ('a', 'r'): 82818,
 ('a', '_'): 81686,
 ('s', 't'): 78706,
 ('n', 'o'): 77134,
 ('e', 's'): 71709,
 ('e', 'a'): 68049,
 ('h', '_'): 64617,
 ('f', '_'): 63686,
 ('m', 'e_'): 62434,
 ('a', 'l'): 61623,
 ('w', 'h'): 61169,
 ('m', '_'): 61166,
 ('d', 'o'): 60236,
 ('e', 'l')

In [529]:
test_vocab2

[[['.', '_'], 435802],
 [[',', '_'], 177291],
 [['y', 'o', 'u', '_'], 134548],
 [['i', '_'], 122767],
 [['t', 'h', 'e_'], 89859],
 [['?', '_'], 84640],
 [['t', 'o', '_'], 73192],
 [['a', '_'], 61661],
 [["'", 's', '_'], 57045],
 [['i', 't', '_'], 55074],
 [["'", 't', '_'], 50411],
 [['t', 'h', 'a', 't', '_'], 43020],
 [['a', 'n', 'd', '_'], 41124],
 [['o', 'f', '_'], 39514],
 [['i', 'n', '_'], 32142],
 [['!', '_'], 30794],
 [['m', 'e_'], 29977],
 [['w', 'h', 'a', 't', '_'], 26868],
 [['i', 's', '_'], 26378],
 [['h', 'e_'], 26149],
 [['f', 'o', 'r', '_'], 23935],
 [['w', 'e_'], 23370],
 [['t', 'h', 'i', 's', '_'], 20366],
 [['y', 'o', 'u', 'r', '_'], 20158],
 [['b', 'e_'], 20137],
 [['m', 'y', '_'], 20105],
 [["'", 'l', 'l', '_'], 20011],
 [['h', 'a', 'v', 'e_'], 19830],
 [['d', 'o', 'n', '_'], 18888],
 [['d', 'o', '_'], 18663],
 [['o', 'n', '_'], 18052],
 [['n', 'o', '_'], 17694],
 [["'", 'm', '_'], 16935],
 [['a', 'l', 'l', '_'], 16776],
 [['b', 'u', 't', '_'], 16067],
 [['w', 'a', 's

In [553]:
tabset = {'e', 'e_'}
tabset.isdisjoint(['t', 'h', 'e', '_'])

False

In [552]:
tab_test = ['t', 'h', 'e', '_']
list(zip(tab_test, tab_test[1:]))

[('t', 'h'), ('h', 'e'), ('e', '_')]

In [545]:
def func_2(vocab, tab):
    pair_counts = defaultdict(int)
    for toks, freq in vocab:
        if not tab.isdisjoint(toks):
            for pair in zip(toks, toks[1:]):
                pair_counts[pair] += freq
    return max(pair_counts, key=pair_counts.get), pair_counts

In [548]:
test_chuj = func_2(test_vocab2, tabset)

In [550]:
test_chuj

(('e', 'r'),
 defaultdict(int,
             {('t', 'h'): 162311,
              ('h', 'e_'): 127255,
              ('m', 'e_'): 62434,
              ('w', 'e_'): 23706,
              ('b', 'e_'): 22903,
              ('h', 'a'): 36549,
              ('a', 'v'): 29656,
              ('v', 'e_'): 52048,
              ("'", 'r'): 15549,
              ('r', 'e_'): 92025,
              ('h', 'e'): 123261,
              ('e', 'r'): 171377,
              ('a', 'r'): 51514,
              ('g', 'e'): 30475,
              ('e', 't'): 53453,
              ('t', '_'): 63504,
              ('w', 'e'): 28681,
              ('e', 'l'): 59120,
              ('l', 'l'): 38015,
              ('l', '_'): 34786,
              ('e', 'y'): 22413,
              ('y', '_'): 55128,
              ("'", 'v'): 11582,
              ('l', 'i'): 38107,
              ('i', 'k'): 12661,
              ('k', 'e_'): 26244,
              ('y', 'e'): 21246,
              ('e', 's'): 71709,
              ('s', '_'): 76876,
 

In [547]:
tabset = {'e', 'e_'}
t = time.time()
for _ in range(100):
    func_2(test_vocab2, tabset)
    pass
print(f"{time.time()-t:.2f} seconds")

3.49 seconds


In [511]:
def proc_count(dct, pair):
    dct_proc = {k: v for k, v in dct.items() if set(pair).isdisjoint(k)}
    return dct_proc

In [515]:
dct_proc1 = proc_count(dct_c1, ('e', '_'))
dct_proc1 = dict(sorted(dct_proc1.items(), key=lambda x: x[1], reverse=True))

In [523]:
dct_proc1

{('t', 'h'): 286912,
 ('o', 'u'): 241694,
 ('i', 'n'): 187051,
 ('y', 'o'): 161552,
 ('a', 'n'): 156511,
 ('h', 'a'): 141491,
 ('o', 'n'): 122342,
 ('a', 't'): 120991,
 ('t', 'o'): 108364,
 ('l', 'l'): 104796,
 ('i', 't'): 104719,
 ('n', 'g'): 99478,
 ('o', 'r'): 88425,
 ('h', 'i'): 86574,
 ('n', 'd'): 86449,
 ('i', 's'): 85924,
 ('a', 'r'): 82818,
 ('s', 't'): 78706,
 ('n', 'o'): 77134,
 ('a', 'l'): 61623,
 ('w', 'h'): 61169,
 ('d', 'o'): 60236,
 ('a', 's'): 58861,
 ('o', 'w'): 57751,
 ("'", 's'): 57136,
 ('u', 'r'): 54896,
 ('h', 'o'): 54478,
 ('u', 't'): 53640,
 ('w', 'a'): 53622,
 ('o', 'm'): 51184,
 ('l', 'i'): 50694,
 ("'", 't'): 50517,
 ('o', 't'): 49292,
 ('g', 'o'): 47979,
 ('n', 't'): 47700,
 ('r', 'i'): 47625,
 ('m', 'a'): 47126,
 ('o', 'f'): 46999,
 ('c', 'o'): 45898,
 ('u', 's'): 43270,
 ('c', 'a'): 42797,
 ('t', 'i'): 42341,
 ('r', 'o'): 42178,
 ('o', 'o'): 42081,
 ('l', 'o'): 41988,
 ('s', 'o'): 39673,
 ('f', 'o'): 39473,
 ('a', 'y'): 39460,
 ('g', 'h'): 37912,
 ('w', 'i

In [518]:
len(dct_c1), len(dct_c2)

(901, 926)

In [462]:
for key in dct_c1.keys():
    if key not in dct_c2.keys():
        print(key)

('e', '_')
('6', 'e')


In [493]:
ile = 0
for key in dct_c2.keys():
    if key not in dct_c1.keys():
        #print(key)
        ile += 1
print(ile)

27


In [509]:
tab1 = [x for x in dct_c1.keys() if set(pair).isdisjoint(x)]
test_1 = [dct_c1[x] == dct_c2[x] for x in tab1]
print(all(test_1))

True


In [488]:
pair = ('e', '_')
len([x for x in dct_c1.keys() if not set(pair).isdisjoint(x)]), len([x for x in dct_c2.keys() if not set(pair).isdisjoint(x)])

(105, 104)

In [459]:
dct_c1

{('e', '_'): 563767,
 ('.', '_'): 435802,
 ('t', '_'): 405188,
 ('t', 'h'): 286912,
 ('s', '_'): 279133,
 ('h', 'e'): 250516,
 ('o', 'u'): 241694,
 ('n', '_'): 224509,
 ('d', '_'): 199163,
 ('i', 'n'): 187051,
 (',', '_'): 177291,
 ('e', 'r'): 171377,
 ('y', '_'): 170823,
 ('r', '_'): 169575,
 ('y', 'o'): 161552,
 ('o', '_'): 158570,
 ('a', 'n'): 156511,
 ('r', 'e'): 150117,
 ('h', 'a'): 141491,
 ('u', '_'): 135623,
 ('i', '_'): 126232,
 ('o', 'n'): 122342,
 ('a', 't'): 120991,
 ('t', 'o'): 108364,
 ('l', '_'): 107287,
 ('l', 'l'): 104796,
 ('i', 't'): 104719,
 ('n', 'g'): 99478,
 ('m', 'e'): 97781,
 ('v', 'e'): 95608,
 ('g', '_'): 89713,
 ('o', 'r'): 88425,
 ('h', 'i'): 86574,
 ('n', 'd'): 86449,
 ('i', 's'): 85924,
 ('e', 'n'): 85895,
 ('?', '_'): 84640,
 ('a', 'r'): 82818,
 ('a', '_'): 81686,
 ('s', 't'): 78706,
 ('n', 'o'): 77134,
 ('e', 's'): 71709,
 ('s', 'e'): 68448,
 ('t', 'e'): 68429,
 ('e', 'a'): 68049,
 ('l', 'e'): 67832,
 ('h', '_'): 64617,
 ('f', '_'): 63686,
 ('a', 'l'): 

In [460]:
dct_c2

{('.', '_'): 435802,
 ('t', '_'): 405188,
 ('t', 'h'): 286912,
 ('s', '_'): 279133,
 ('o', 'u'): 241694,
 ('n', '_'): 224509,
 ('d', '_'): 199163,
 ('i', 'n'): 187051,
 (',', '_'): 177291,
 ('e', 'r'): 171377,
 ('y', '_'): 170823,
 ('r', '_'): 169575,
 ('y', 'o'): 161552,
 ('o', '_'): 158570,
 ('a', 'n'): 156511,
 ('h', 'a'): 141491,
 ('u', '_'): 135623,
 ('h', 'e_'): 127255,
 ('i', '_'): 126232,
 ('h', 'e'): 123261,
 ('o', 'n'): 122342,
 ('a', 't'): 120991,
 ('t', 'o'): 108364,
 ('l', '_'): 107287,
 ('l', 'l'): 104796,
 ('i', 't'): 104719,
 ('n', 'g'): 99478,
 ('r', 'e_'): 92025,
 ('g', '_'): 89713,
 ('o', 'r'): 88425,
 ('h', 'i'): 86574,
 ('n', 'd'): 86449,
 ('i', 's'): 85924,
 ('e', 'n'): 85895,
 ('?', '_'): 84640,
 ('a', 'r'): 82818,
 ('a', '_'): 81686,
 ('s', 't'): 78706,
 ('n', 'o'): 77134,
 ('e', 's'): 71709,
 ('e', 'a'): 68049,
 ('h', '_'): 64617,
 ('f', '_'): 63686,
 ('m', 'e_'): 62434,
 ('a', 'l'): 61623,
 ('w', 'h'): 61169,
 ('m', '_'): 61166,
 ('d', 'o'): 60236,
 ('e', 'l')

#### 2.VER -  MOZNA ODEJMOWAC OD TYCH CO ZNALAZLA: ('h', 'e'): 250516 - ('h', 'e_'): 127255 ==> ('h', 'e'): 123261

#### 3.4 BPE Encoder propably should add to the BPE Class Later

In [107]:
test_word = ' '.join(char_factor("running"))
test_word

'r u n n i n g </w>'

In [175]:
# pair = 'g </w>'
pair = '['
sub_pair = lambda x: re.sub('\s+', '', x)
re.sub(rf"( |^){re.escape(pair)}( |$)", rf"\g<1>{sub_pair(pair)}\g<2>", "r u n n i n g </w>")

'r u n n i n g </w>'

In [196]:
def word_encoder(word, vocab):
    word_str = ' '.join(char_factor(word)) #do zmiany uzywanie char_factor spoza funkcji
    sub_key = lambda x: re.sub('\s+', '', x)
    
    for k, v in vocab.items():
        if bool(re.search(rf"( |^){re.escape(k)}( |$)", word_str)):
            print(k, v)
            print(word_str)
            word_str = re.sub(rf"( |^){re.escape(k)}( |$)", rf"\g<1>{sub_key(k)}\g<2>", word_str)
            print(word_str, '\n')
    return word_str

In [218]:
tokenizer_eng.vocab

{'<pad>': 0,
 '<unk>': 1,
 '<eos>': 2,
 '</w>': 3,
 '9': 4,
 'h': 5,
 '4': 6,
 'z': 7,
 '.': 8,
 't': 9,
 '%': 10,
 '1': 11,
 'u': 12,
 'w': 13,
 '7': 14,
 '[': 15,
 '3': 16,
 'c': 17,
 '5': 18,
 'd': 19,
 'i': 20,
 '8': 21,
 '$': 22,
 ']': 23,
 ')': 24,
 'v': 25,
 'l': 26,
 'k': 27,
 ',': 28,
 'm': 29,
 'r': 30,
 'j': 31,
 'q': 32,
 's': 33,
 '"': 34,
 '6': 35,
 'x': 36,
 '?': 37,
 'e': 38,
 'f': 39,
 'y': 40,
 '/': 41,
 '(': 42,
 '-': 43,
 ':': 44,
 'o': 45,
 '2': 46,
 "'": 47,
 '0': 48,
 'g': 49,
 'n': 50,
 'p': 51,
 '!': 52,
 'a': 53,
 'b': 54,
 'e </w>': 55,
 '. </w>': 56,
 't </w>': 57,
 't h': 58,
 's </w>': 59,
 'o u': 60,
 'n </w>': 61,
 'd </w>': 62,
 ', </w>': 63,
 'e r': 64,
 'y </w>': 65,
 'y ou': 66,
 'o </w>': 67,
 'i n': 68,
 'you </w>': 69,
 'i </w>': 70,
 'a n': 71,
 'l </w>': 72,
 'r </w>': 73,
 'th e</w>': 74,
 'g </w>': 75,
 'a t</w>': 76,
 '? </w>': 77,
 'a </w>': 78,
 'l l</w>': 79,
 't o</w>': 80,
 'in g</w>': 81,
 'a r': 82,
 'f </w>': 83,
 'i t</w>': 84,
 'm e

In [333]:
def encode_word(word, vocab):
    word_str = ' '.join(word)
    sub_key = lambda x: re.sub('\s+', '', x)
    for key in vocab.keys():
        pat = ' ' + key + ' '
        if pat in word_str:
            print(key)
            print(word_str, '\n')
            word_str = word_str.replace(key, sub_key(key))
            # print(key)
            # print(word_str, '\n')
    return word_str

In [336]:
encode_word(char_factor(' running '), tokenizer_eng.vocab)

n
  r u n n i n g   </w> 

g
  r u n n i n g   </w> 

r
  r u n n i n g   </w> 

u
  r u n n i n g   </w> 

i
  r u n n i n g   </w> 

i n
  r u n n i n g   </w> 

u n
  r u n n in g   </w> 



'  r un n in g   </w>'

In [269]:
test_snt = "Yesterday I visited zoo with my family, it was super fun!"
test_snt_split = tokenize_eng(test_snt)
print(test_snt_split)

['yesterday', 'i', 'visited', 'zoo', 'with', 'my', 'family', ',', 'it', 'was', 'super', 'fun', '!']


In [270]:
[[x for x in word] + ['</w>'] for word in test_snt_split]

[['y', 'e', 's', 't', 'e', 'r', 'd', 'a', 'y', '</w>'],
 ['i', '</w>'],
 ['v', 'i', 's', 'i', 't', 'e', 'd', '</w>'],
 ['z', 'o', 'o', '</w>'],
 ['w', 'i', 't', 'h', '</w>'],
 ['m', 'y', '</w>'],
 ['f', 'a', 'm', 'i', 'l', 'y', '</w>'],
 [',', '</w>'],
 ['i', 't', '</w>'],
 ['w', 'a', 's', '</w>'],
 ['s', 'u', 'p', 'e', 'r', '</w>'],
 ['f', 'u', 'n', '</w>'],
 ['!', '</w>']]

In [135]:
def connect_pairs(vocab_chars, pair):
    vocab_proc = vocab_chars.copy()
    pair_con = re.sub('\s+', '', pair)
    for i, (toks, _) in enumerate(vocab_proc):
        toks = " ".join(toks).replace(pair, pair_con).split()
        vocab_proc[i][0] = toks
    return vocab_proc

In [221]:
def most_common_pair(vocab_chars):
    pair_counts = Counter()
    for toks, freq in vocab_chars:
        for pair in map(' '.join, zip(toks, toks[1:])):
            pair_counts[pair] += freq
    return pair_counts.most_common(1)[0][0]

In [226]:
vocab_chars_eng

[[['.', '</w>'], 435802],
 [[',', '</w>'], 177291],
 [['y', 'o', 'u', '</w>'], 134548],
 [['i', '</w>'], 122767],
 [['t', 'h', 'e', '</w>'], 89859],
 [['?', '</w>'], 84640],
 [['t', 'o', '</w>'], 73192],
 [['a', '</w>'], 61661],
 [["'", 's', '</w>'], 57045],
 [['i', 't', '</w>'], 55074],
 [["'", 't', '</w>'], 50411],
 [['t', 'h', 'a', 't', '</w>'], 43020],
 [['a', 'n', 'd', '</w>'], 41124],
 [['o', 'f', '</w>'], 39514],
 [['i', 'n', '</w>'], 32142],
 [['!', '</w>'], 30794],
 [['m', 'e', '</w>'], 29977],
 [['w', 'h', 'a', 't', '</w>'], 26868],
 [['i', 's', '</w>'], 26378],
 [['h', 'e', '</w>'], 26149],
 [['f', 'o', 'r', '</w>'], 23935],
 [['w', 'e', '</w>'], 23370],
 [['t', 'h', 'i', 's', '</w>'], 20366],
 [['y', 'o', 'u', 'r', '</w>'], 20158],
 [['b', 'e', '</w>'], 20137],
 [['m', 'y', '</w>'], 20105],
 [["'", 'l', 'l', '</w>'], 20011],
 [['h', 'a', 'v', 'e', '</w>'], 19830],
 [['d', 'o', 'n', '</w>'], 18888],
 [['d', 'o', '</w>'], 18663],
 [['o', 'n', '</w>'], 18052],
 [['n', 'o', '</

In [102]:
def most_common_pair(vocab_chars):
    freq_pair = defaultdict(int)
    for toks, freq in vocab_chars:
        toks_pairs = [" ".join(pair) for pair in zip(toks, toks[1:])]
        for pair in toks_pairs:
            freq_pair[pair] += freq
    return max(freq_pair.items(), key=lambda x: x[1])[0]

In [190]:
pair_counts = Counter()
pair_counts[('t','h')] += 10
pair_counts[('t','h')] += 10
pair_counts.most_common(1)

[(('t', 'h'), 20)]

In [394]:
vocab_chars_eng2 = [[char_factor(k), v] for k, v in uniq_freq_eng.most_common()]
tokenizer_eng_test = BPETokenizer(vocab_chars_eng2, 100, False)
t = time.time()
tokenizer_eng_test.train_bpe()
print(f"{time.time()-t:.2f} seconds")

4.49 seconds
